In [17]:
import pandas as pd

columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]

base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/"
hospitals = ["cleveland", "hungarian", "switzerland", "va"]

frames = []
for h in hospitals:
    part = pd.read_csv(base_url + f"processed.{h}.data", header=None,
                       names=columns, na_values="?")
    part["source"] = h
    frames.append(part)

df = pd.concat(frames, ignore_index=True)

print(df.shape)
print(df["source"].value_counts())
print(df.isna().sum())

(920, 15)
source
cleveland      303
hungarian      294
va             200
switzerland    123
Name: count, dtype: int64
age           0
sex           0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalach      55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
source        0
dtype: int64


In [18]:
import numpy as np 
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])
df[["chol", "trestbps"]] = df[["chol", "trestbps"]].replace(0, np.nan)

print(df["target"].value_counts())
print((df.drop(columns=["source", "target"]).isna()
         .groupby(df["source"]).mean() * 100).round(0))

target
1    509
0    411
Name: count, dtype: int64
             age  sex   cp  trestbps   chol   fbs  restecg  thalach  exang  \
source                                                                       
cleveland    0.0  0.0  0.0       0.0    0.0   0.0      0.0      0.0    0.0   
hungarian    0.0  0.0  0.0       0.0    8.0   3.0      0.0      0.0    0.0   
switzerland  0.0  0.0  0.0       2.0  100.0  61.0      1.0      1.0    1.0   
va           0.0  0.0  0.0      28.0   28.0   4.0      0.0     26.0   26.0   

             oldpeak  slope    ca  thal  
source                                   
cleveland        0.0    0.0   1.0   1.0  
hungarian        0.0   65.0  99.0  90.0  
switzerland      5.0   14.0  96.0  42.0  
va              28.0   51.0  99.0  83.0  


In [21]:
from sklearn.model_selection import train_test_split

SEED = 7   # change to your own number

X = df.drop(columns=["target", "source"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

test_df = X_test.copy()
test_df["target"] = y_test.values
test_df.to_csv("test_data.csv", index=False)
print(X_train.shape, X_test.shape)


(736, 13) (184, 13)
